# Load Dependencies


## Files needed to run:

- `feature_columns.pkl`
- `svm_finetuned.pkl`
- `rf_finetuned.pkl`
- `CBERT_checkpoint.pth`
- `EnsembleProcessing.py`
- All files are available in the Final Implementation folder of our [repository](https://github.com/MiguelPartosa/Thesis-FOS-BinaryClass-WSD).


In [1]:
import pandas as pd
import torch
from torch import cuda
from transformers import DistilBertTokenizer, DistilBertModel

c:\Users\User\miniconda3\envs\data_science\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## CBERT Checkpoint


In [2]:
device = 'cuda' if cuda.is_available() else 'cpu'


class BertClass(torch.nn.Module):
    def __init__(self):
        super(BertClass, self).__init__()
        self.l1 = DistilBertModel.from_pretrained('GianTan/CBERTo')
        self.pre_classifier = torch.nn.Linear(768, 768)
        self.dropout = torch.nn.Dropout(0.3)

        self.pre_classifier2 = torch.nn.Linear(768, 768)
        self.dropout2 = torch.nn.Dropout(0.3)

        self.classifier = torch.nn.Linear(768, 1)

    def forward(self, input_ids, attention_mask, token_type_ids):
        output_1 = self.l1(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = output_1[0]
        pooler = hidden_state[:, 0]
        pooler = self.pre_classifier(pooler)
        pooler = torch.nn.Tanh()(pooler)
        pooler = self.dropout(pooler)
        pooler = self.pre_classifier2(pooler)
        pooler = torch.nn.Tanh()(pooler)
        pooler = self.dropout2(pooler)

        output = self.classifier(pooler)
        return output.squeeze(1)


cbert_model = BertClass()
cbert_model.to(device)

tokenizer = DistilBertTokenizer.from_pretrained(
    'GianTan/CBERTo', truncation=True, do_lower_case=False)
optimizer = torch.optim.Adam(params=cbert_model.parameters(), lr=4e-05)

# load
checkpoint = torch.load('CBERT_checkpoint.pth', map_location='cpu')
cbert_model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

## SVM and Random Forest Checkpoint


Importing


In [3]:
import pickle
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

# Dependencies from read pickle file


def GetTextCol(X):
    text_cols = X.select_dtypes(include=['object', 'string']).columns
    if len(text_cols) == 0:
        raise ValueError("No text columns found in input DataFrame")
    # Error is raised if the first index is not returned.
    return text_cols[0]


def GetNumCol(X):
    return X.select_dtypes(include=['int64', 'float64']).columns.tolist()


preprocessor = ColumnTransformer(
    transformers=[
        ("tfidf", TfidfVectorizer(), GetTextCol),
        ("scaler", StandardScaler(), GetNumCol)
    ],
    remainder='drop'  # Remove any unhandled columns
)

# SVM
with open('svm_finetuned.pkl', 'rb') as f:
    svm_best = pickle.load(f)

# Random Forest
with open('rf_finetuned.pkl', 'rb') as f:
    rf_best = pickle.load(f)

c:\Users\User\miniconda3\envs\data_science\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\User\miniconda3\envs\data_science\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\User\miniconda3\envs\data_science\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.

# Model Function Backend


## CBERT


In [4]:
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# Unchanged function from cbert train


class FOS_Classification(Dataset):

    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.data = dataframe
        self.text = dataframe.Usage
        self.targets = self.data.is_fos
        self.max_len = max_len

    def __len__(self):
        return len(self.text)

    def __getitem__(self, index):
        text = str(self.text[index])
        text = " ".join(text.split())

        inputs = self.tokenizer.encode_plus(
            text,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            truncation=True,
            pad_to_max_length=True,
            return_token_type_ids=True
        )
        ids = inputs['input_ids']
        mask = inputs['attention_mask']
        token_type_ids = inputs["token_type_ids"]

        return {
            'ids': torch.tensor(ids, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.long),
            'token_type_ids': torch.tensor(token_type_ids, dtype=torch.long),
            'targets': torch.tensor(self.targets[index], dtype=torch.float)
        }

In [5]:
# Function taken from Cbert train file to calculate multiple predictions from a dataframe

# Hyperparmeters loaded from original
test_params = {'batch_size': 128,
               'shuffle': True,
               'num_workers': 0
               }

# Load dataframe into prereq functions


def validation(df):
    '''
    Input df to return preds in array. 
    ## Requires:
    - 'is_fos' column with values (0,1) to work. 
    - 'Usage' column with the sentence to be processed.

    ## Outputs:
    - We are only concerened with the fin_outputs for experimentation usage
    '''
    testing_set = FOS_Classification(df, tokenizer, 256)
    testing_loader = DataLoader(testing_set, **test_params)
    cbert_model.eval()
    fin_targets = []
    fin_outputs = []

    with torch.no_grad():
        for _, data in tqdm(enumerate(testing_loader, 0)):
            ids = data['ids'].to(device, dtype=torch.long)
            mask = data['mask'].to(device, dtype=torch.long)
            token_type_ids = data['token_type_ids'].to(
                device, dtype=torch.long)
            targets = data['targets'].to(device, dtype=torch.float)
            outputs = cbert_model(ids, mask, token_type_ids)
            fin_targets.extend(targets.cpu().detach().numpy().tolist())
            fin_outputs.extend(torch.sigmoid(
                outputs).cpu().detach().numpy().tolist())

    # Added threshhold of 0.9 here to avoid having to do it after every call
    fin_outputs = [1 if x >= 0.9 else 0 for x in fin_outputs]

    return fin_outputs, fin_targets

## SVM and Random Forest


In [27]:
# When predicting on new data:
from EnsembleProcessing import process_embeddings
import ast
# Fix Reloading issue when fixing from process_embeddings
from importlib import reload
import EnsembleProcessing
reload(EnsembleProcessing)

# Dataframe needed to merge with function to acquire needed verbs
# Filtereing only to needed verbs
original_df = pd.read_excel('./../Dataset/Final_Dataset_WithLabels.xlsx')
original_df = original_df.drop_duplicates(subset='Verb').reset_index(drop=True)
# Fix stringed lists to list to joined string
original_df['Verb'] = original_df['Verb'].apply(
    lambda x: ','.join(ast.literal_eval(x)))



def PredictSample(input_df):
    '''
    Function receives dataframe and returns predictions from random forest and svm
    ## Requirements
    - Columns: 'Word Sense', 'Verb', and 'Usage'(the sentence)
    '''
    global original_df
    merge_df = original_df.merge(input_df, on='Verb', how='right')
    if merge_df.isna().sum().sum() > 0:
        display('Rows from merged dataframe with null values:',merge_df[merge_df.isna().any(axis=1)])
        raise Exception('NA values present in merged dataset of input dataframe and the original dataframe. Possible mismatch of verb values.')
    merge_df['Sentence Type'] = merge_df['Sentence Type'].apply(
        lambda x: 0 if x == 'Literal' else 1)
    merge_df.drop(columns=['Usage', 'Is FOS'], inplace=True)

    # Renaming columns because original implementation(process_embeddings) required these column names to generate embeddings
    merge_df.rename(columns={'Sentence': 'Usage',
                    'Sentence Type': 'Is FOS'}, inplace=True)
    input_df = merge_df

    # Load models and feature columns

    # SVM
    with open('svm_finetuned.pkl', 'rb') as f:
        svm_model = pickle.load(f)

    # RF
    with open('rf_finetuned.pkl', 'rb') as f:
        rf_model = pickle.load(f)

    # features for processing of dataframses
    with open('feature_columns.pkl', 'rb') as f:
        feature_columns = pickle.load(f)
        
    # Embeddings
    embeddings_df = process_embeddings(input_df, variance_threshold=1)
    combined_df = pd.concat([embeddings_df, input_df], axis=1)
    # Clean read issue tensor values
    if 'Similarity Scores' in combined_df.columns:
        combined_df['Similarity Scores'] = combined_df['Similarity Scores'].apply(
            lambda x: x.item() if hasattr(x, 'item') else x
        )
        
    # Drop Non-features
    for col in ['Is FOS', 'Word Sense', 'Verb', 'Usage']:
        if col in combined_df.columns:
            combined_df = combined_df.drop(columns=[col])

    # Align
    # Create a DataFrame with all required columns, filled with zeros
    aligned_row = pd.DataFrame(
        0, index=combined_df.index, columns=feature_columns)

    for col in combined_df.columns:
        if col in feature_columns:
            aligned_row[col] = combined_df[col]

    display(aligned_row)
    rf_prob = rf_model.predict_proba(aligned_row)
    rf_prob = [1 if prob[1] >= 0.5 else 0 for prob in rf_prob]
    svm_pred = svm_model.predict(aligned_row)
    
    return {
        'rf_prediction': rf_prob,
        'svm_prediction': svm_pred.tolist()
    }
# sample_1 = PredictSample(example_output)
# print("Sample Output with fake data:")
# sample_1

## All Models


In [7]:
import warnings


def prediction_values(score: list) -> list:
    prediction_results = []
    for predict in score:
        prediction_results.append('Non-Literal' if predict >= 0.5 else 'Literal')
    return prediction_results


def ClassifyRows(df) -> dict:
    '''
    Returns:
    - Dict for classification results in a list
    - *Keys:* 'CBERT', 'SVM' , and 'RF' 

    '''
    # CBERT
    # Pre-req for processing
    cbert_df = df.copy()
    cbert_df['is_fos'] = cbert_df['Sentence Type'].apply(
        lambda x: 0 if x == 'Literal' else 1)
    cbert_df.rename(columns={'Sentence': 'Usage'}, inplace=True)

    # Results of cbert, fin_outputs(0)
    cbert_result = validation(cbert_df)[0]

    # Ignores appear from old impl.
    with warnings.catch_warnings(action="ignore"):
        models_result = PredictSample(df)
    svm_result = models_result['svm_prediction']
    rf_result = models_result['rf_prediction']
    return {'CBERT': prediction_values(cbert_result), 'SVM': prediction_values(svm_result), 'RF': prediction_values(rf_result)}
    # return {'CBERT': cbert_result, 'SVM': svm_result, 'RF': rf_result}

# Experimentation

## Transforming Dataset

- Transform three input columns into set format for function of predicting usage by merging with their word sense from the original dataset


In [8]:
example_df = pd.read_excel(
    './Experimentation Implementation/User_Testing (1).xlsx')
example_df

,Verbs,Literal Example,Non-Literal Example
0,magbiko,Magbiko ta karong hapon para sa pista,Nagbiko na pud siya sa iyang desisyon.
1,naay,Naay isda sa balde.,Naay siya’y kasing-kasing nga maayong buhatonon.
2,manakop,Manglakaw ta aron manakop og isda sa suba.,Manakop na pud ang pulis sa mga walay lisensya
3,namatyag,Nagpabilin siya sa balay kay namatyag siya sa ...,Namatyag siya nga naay nagtan-aw niya.
4,mag-ugbok,Dapat mag-ugbok ka sa humay aron maluto pag-ayo.,Nag-ugbok siya sa kasubo sa pagkawala sa iyang...
...,...,...,...
70,manira,Manira ang tindahan alas nuwebe sa gabii.,Manira siya sa iyang kaugalingong kalipay tung...
71,mohalang,Mohalang siya sa agianan sa trak.,Mohalang siya sa akong damgo nga dugay nang guba.
72,nagluno,Nagluno siya sa salog samtang nagtan-aw og TV.,Nagluno siya sa kasakit nga dugay na niyang gi...
73,nakatugdon,Nakatugdon siya og hapsay nga yuta para sa balay.,Nakatugdon siya sa tinuod niyang tinguha sa ki...


In [9]:
def TransformDataset(df, user_id: str) -> pd.DataFrame:
    # Prep output dataset
    transformed_columns = ['User ID', 'Verb', 'Sentence Type', 'Sentence']
    transformed_df = pd.DataFrame(columns=transformed_columns)

    # Process dataset into examples. 30 rows result
    for row in example_df.itertuples():
        transformed_df = pd.concat([transformed_df,
                                    pd.DataFrame({
                                        'User ID': user_id, 'Verb': row[1], 'Sentence Type': 'Literal', 'Sentence': row[2]}, index=[row[0]*2]),
                                    pd.DataFrame(
                                        {'User ID': user_id, 'Verb': row[1], 'Sentence Type': 'Non-Literal', 'Sentence': row[3]}, index=[row[0]*2+1])
                                    ])
    return transformed_df


example_output = TransformDataset(example_df, 'Mama ni Giordan')
example_output

,User ID,Verb,Sentence Type,Sentence
0,Mama ni Giordan,magbiko,Literal,Magbiko ta karong hapon para sa pista
1,Mama ni Giordan,magbiko,Non-Literal,Nagbiko na pud siya sa iyang desisyon.
2,Mama ni Giordan,naay,Literal,Naay isda sa balde.
3,Mama ni Giordan,naay,Non-Literal,Naay siya’y kasing-kasing nga maayong buhatonon.
4,Mama ni Giordan,manakop,Literal,Manglakaw ta aron manakop og isda sa suba.
...,...,...,...,...
145,Mama ni Giordan,nagluno,Non-Literal,Nagluno siya sa kasakit nga dugay na niyang gi...
146,Mama ni Giordan,nakatugdon,Literal,Nakatugdon siya og hapsay nga yuta para sa balay.
147,Mama ni Giordan,nakatugdon,Non-Literal,Nakatugdon siya sa tinuod niyang tinguha sa ki...
148,Mama ni Giordan,gitilapan,Literal,Gitilapan sa iring ang platito nga adunay gatas.


In [10]:
# show_result_df = example_output.copy()
# show_result_df['is_fos'] = show_result_df['Sentence Type'].apply(lambda x: 0 if x == 'Literal' else 1)
# show_result_df.rename(columns={'Sentence' : 'Usage'}, inplace=True)
# result_sample = validation(show_result_df)
# result_sample[0]

In [28]:
import numpy as np
model_predictions = ClassifyRows(example_output)

predict_output_cols = {'CBERT Prediction': model_predictions['CBERT'],
                       'SVM Prediction': model_predictions['SVM'], 'Random Forest Prediction': model_predictions['RF']}
for key in predict_output_cols:
    predict_output_cols[key] = pd.Series(predict_output_cols[key]).reindex(example_output.index, fill_value=np.nan)

example_output = example_output.assign(**predict_output_cols)
example_output

0it [00:00, ?it/s]c:\Users\User\miniconda3\envs\data_science\Lib\site-packages\transformers\tokenization_utils_base.py:2700: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(
2it [00:01,  1.51it/s]


'Rows from merged dataframe with null values:'

,FOS,Word Sense,Verb,Usage,Is FOS,User ID,Sentence Type,Sentence
2,NaN,NaN,naay,NaN,NaN,Mama ni Giordan,Literal,Naay isda sa balde.
3,NaN,NaN,naay,NaN,NaN,Mama ni Giordan,Non-Literal,Naay siya’y kasing-kasing nga maayong buhatonon.
18,NaN,NaN,pasulti-on,NaN,NaN,Mama ni Giordan,Literal,Pasulti-on nato ang bata sa iyang pangalan.
19,NaN,NaN,pasulti-on,NaN,NaN,Mama ni Giordan,Non-Literal,Dili lang sila pasulti-on sa ilang tinuod nga ...
22,NaN,NaN,pabuhaton,NaN,NaN,Mama ni Giordan,Literal,Pabuhaton nako ang akong anak sa iyang assignment
23,NaN,NaN,pabuhaton,NaN,NaN,Mama ni Giordan,Non-Literal,Pabuhaton siya sa desisyon nga dili niya gusto
34,NaN,NaN,masipyat,NaN,NaN,Mama ni Giordan,Literal,masipyat man jud ta kay tawo raman ta
35,NaN,NaN,masipyat,NaN,NaN,Mama ni Giordan,Non-Literal,masipyat gani ang karnero sa pagsalig sa lobo
42,NaN,NaN,makabuhi,NaN,NaN,Mama ni Giordan,Literal,pwede naka makabuhi sa pisi
43,NaN,NaN,makabuhi,NaN,NaN,Mama ni Giordan,Non-Literal,dili siya mosugot nimo kung dili ka makabuhi s...


Exception: NA values present in merged dataset of input dataframe and the original dataframe. Possible mismatch of verb values.

In [ ]:
# TODO Change Everything for experiment implementation, make upoading possible for entire sheet 
# @title Enter new parameters and rerun cell to refresh results. { display-mode: "form" }

# from IPython.display import HTML, display, Javascript

# test_fos = 'makabuhi og patay'  # @param {type:"string"}
# # @param {type:"string"}
# test_word_sense = 'usa ka makapatikod nga bakak o sugilanon'
# test_verb = 'Makabuhi, Patay'  # @param {type:"string"}

# # @param {type:"string"}
# test_usage = 'Ang mga tawo nga makabuhi og patay kasagaran maayo kaayo mamatay.'
# # @param ["Non-literal Example", "Literal Example"]
# test_is_fos = 'Non-literal Example'

# test_is_fos = 0 if test_is_fos == "Literal Example" else 1

# test_df = pd.DataFrame({'FOS': [test_fos], 'Word Sense': [test_word_sense], 'Verb': [
#                        test_verb], 'Usage': [test_usage], 'Is FOS': [test_is_fos]})

# prediction_output = ClassifyRows(test_df)
# print('\n\n')
# df = pd.DataFrame({
#     "Model": ["Cbert", "SVM", "Random Forest"],
#     "Prediction": prediction_output
# })

# html = f"""
#             <div style="scale: 100%;">
#                 {df.to_html(index=False, border=1)}
#             </div>
#         """

# display(HTML(html))
# Javascript("google.colab.output.setIframeHeight('500px');")

c:\Users\Miguel\Documents\Project Source Files\IT Work\School\Thesis\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:2834: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


Generating embeddings...


Transforming Usage Embeddings: 100%|██████████| 3/3 [00:00<00:00, 87.96it/s]


Number of components for 100% variance:
Verb: 768 components
Usage: 768 components
Sentence: 768 components
Optimal number of clusters:
Verb: 2 clusters
Usage: 2 clusters
Sentence: 2 clusters





Model,Prediction
Cbert,Non-Literal
SVM,Non-Literal
Random Forest,Literal


<IPython.core.display.Javascript object>